In [1]:
import torch
import torch.nn as nn

print(torch.__version__)

2.13.0


In [2]:
''' 
TODO: 

Modify the encoder so Seq2Seq retains encoder_outputs.
Implement a standalone BahdanauAttention module.
Test only its shapes and normalization.
Integrate the context vector into one decoder step.
Run full teacher-forced decoding.
Add greedy autoregressive generation.
Inspect attention matrices on a tiny memorization task.

''';

In [83]:
SRC_VOCAB_SIZE = 10
TGT_VOCAB_SIZE = 10

SRC_LENGTH = 3
TGT_LENGTH = 3

BATCH_SIZE = 5

EMBEDDING_DIM = 32
ATTENTION_DIM = 20

ENC_HIDDEN_DIM = 48
DEC_HIDDEN_DIM = 64

NUM_LAYERS = 2

SOS_IDX = 1
EOS_IDX = 2

In [84]:
src = torch.randint(
    low=3,
    high=SRC_VOCAB_SIZE,
    size=(BATCH_SIZE, SRC_LENGTH)
)

tgt = src.clone().detach()

SOS_col = torch.tensor([SOS_IDX])
SOS_col = SOS_col.expand(BATCH_SIZE, -1)

EOS_col = torch.tensor([EOS_IDX])
EOS_col = EOS_col.expand(BATCH_SIZE, -1)

tgt = torch.cat((SOS_col, src, EOS_col), dim=1)

src, tgt

(tensor([[8, 5, 5],
         [4, 7, 3],
         [5, 3, 3],
         [9, 8, 8],
         [6, 8, 4]]),
 tensor([[1, 8, 5, 5, 2],
         [1, 4, 7, 3, 2],
         [1, 5, 3, 3, 2],
         [1, 9, 8, 8, 2],
         [1, 6, 8, 4, 2]]))

In [101]:
class Encoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        enc_hidden_dim,
        dec_hidden_dim,
        num_layers,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        self.lstm = nn.LSTM(
            input_size = embedding_dim,
            hidden_size = enc_hidden_dim,
            num_layers = num_layers,
            batch_first=True
        )

        self.encoder_decoder_bridge = nn.Linear(
            in_features=enc_hidden_dim,
            out_features=dec_hidden_dim,
        )
        
    def forward(
        self,
        src: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]: # type hint
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.lstm(embedded)

        hidden = self.encoder_decoder_bridge(hidden)
        cell = self.encoder_decoder_bridge(cell)
        
        return outputs, hidden, cell


In [107]:
encoder = Encoder(
    SRC_VOCAB_SIZE,
    EMBEDDING_DIM,
    ENC_HIDDEN_DIM,
    DEC_HIDDEN_DIM,
    NUM_LAYERS
)

encoder_outputs, encoder_hidden, encoder_cell = encoder(src)

encoder_outputs.shape, encoder_hidden.shape, encoder_cell.shape

(torch.Size([5, 3, 48]), torch.Size([2, 5, 64]), torch.Size([2, 5, 64]))

In [ ]:
'''
Example pipeline

decoder_hidden                      (2, 4, 64)
decoder_hidden[-1]                  (4, 64)
unsqueeze(1)                        (4, 1, 64)
query projection                    (4, 1, 20)

encoder_outputs                     (4, 3, 64)
encoder projection                  (4, 3, 20)

add + tanh                          (4, 3, 20)
Linear(20 -> 1)                     (4, 3, 1)
squeeze(-1)                         (4, 3)
softmax(dim=1)                      (4, 3)

unsqueeze weights                   (4, 3, 1)
multiply by encoder_outputs         (4, 3, 64)
sum(dim=1)                          (4, 64)

embedded previous token: (4, 1, 32)
context.unsqueeze(1):    (4, 1, 64)

decoder_input:  (4, 1, 96)
LSTM output:    (4, 1, 64) -> Linear layer -> (4, 1, target_vocab_size)
new hidden:     (2, 4, 64)
new cell:       (2, 4, 64)


input features per step:   96
hidden representation:     64

decoder output:     (4, 1, 64)
linear projection:  64 -> V

logits:      (4, 1, V)
argmax(-1):  (4, 1)
squeeze(1):  (4,)

''';

In [103]:
class Bahdanau(nn.Module):
    def __init__(
        self,
        decoder_hidden_dim: int,
        encoder_hidden_dim: int,
        attention_dim: int,
    ) -> None: 
        super().__init__()
        
        self.project_query = nn.Linear(
            in_features=decoder_hidden_dim,
            out_features=attention_dim,
        )

        self.project_encoder_outputs = nn.Linear(
            in_features=encoder_hidden_dim,
            out_features=attention_dim,
        )

        self.project_energy = nn.Linear(
            in_features=attention_dim,
            out_features=1
        )
    
    def forward(
        self,
        query: torch.Tensor, # batch_size, hidden_dim
        encoder_outputs: torch.Tensor, # batch_size, src_len, hidden_dim,
    ) -> tuple[torch.Tensor, torch.Tensor]:

        projected_query = self.project_query(query).unsqueeze(1) # batch_size, 1, attention_dim
        projected_encoder_outputs = self.project_encoder_outputs(encoder_outputs) # batch_size, src_len, attention_dim

        energy = torch.tanh(projected_query + projected_encoder_outputs) # batch_size, src_len, attention_dim
        scores = self.project_energy(energy).squeeze(-1) # batch_size, src_len

        weights = torch.softmax(scores, dim=1) # batch_size, src_len
        unsqueezed_weights =  weights.unsqueeze(2) # batch_size, src_len, 1
        
        context = unsqueezed_weights * encoder_outputs # batch_size, src_len, hidden_dim
        context = context.sum(dim=1) # batch_size, hidden_dim

        return context, weights

        

In [104]:
bahdanau = Bahdanau(
    decoder_hidden_dim=DEC_HIDDEN_DIM,
    encoder_hidden_dim=ENC_HIDDEN_DIM,
    attention_dim=ATTENTION_DIM,
)

context, weights = bahdanau(
    query=decoder_hidden,
    encoder_outputs=encoder_outputs,
)

assert context.shape == (BATCH_SIZE, ENC_HIDDEN_DIM)
assert weights.shape == (BATCH_SIZE, SRC_LENGTH)
assert torch.allclose(
    weights.sum(dim=1),
    torch.ones(BATCH_SIZE),
    atol=1e-6,
)

context.shape, weights.shape

(torch.Size([5, 48]), torch.Size([5, 3]))

In [105]:
class Decoder(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int,
        encoder_hidden_dim: int,
        decoder_hidden_dim: int,
        num_layers: int,
    ) -> None:
        super().__init__()

        input_dim = embedding_dim + encoder_hidden_dim
        
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
        )

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=decoder_hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )

        self.output = nn.Linear(
            in_features=decoder_hidden_dim,
            out_features=vocab_size,
        )

    def forward(
        self,
        input_token: torch.Tensor,
        context: torch.Tensor,
        hidden: torch.Tensor,
        cell: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:

        embedded = self.embedding(input_token).unsqueeze(1)
        context = context.unsqueeze(1)
        
        lstm_input = torch.cat((embedded, context), dim=-1)
        
        outputs, (hidden, cell) = self.lstm(
            lstm_input,
            (hidden, cell),
        )
        outputs = outputs.squeeze(1)
        logits = self.output(outputs)

        return logits, hidden, cell


In [113]:
decoder = Decoder(
    vocab_size=TGT_VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    encoder_hidden_dim=ENC_HIDDEN_DIM,
    decoder_hidden_dim=DEC_HIDDEN_DIM,
    num_layers=NUM_LAYERS,
)

input_token = torch.full(
            (BATCH_SIZE,),
            SOS_IDX,
            dtype=torch.long,
            device=src.device
        )

logits, hidden, cell = decoder(input_token, context, encoder_hidden, encoder_cell)

assert logits.shape == (BATCH_SIZE, TGT_VOCAB_SIZE)
assert hidden.shape == (NUM_LAYERS, BATCH_SIZE, DEC_HIDDEN_DIM)
assert cell.shape == (NUM_LAYERS, BATCH_SIZE, DEC_HIDDEN_DIM)

logits.shape, hidden.shape, cell.shape

(torch.Size([5, 10]), torch.Size([2, 5, 64]), torch.Size([2, 5, 64]))

In [114]:
class Seq2Seq(nn.Module):
    
    def __init__(
        self,
        encoder: Encoder,
        decoder: Decoder,
        bahdanau: Bahdanau,
        target_vocab_size: int,
    ):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.bahdanau = bahdanau
        self.target_vocab_size = target_vocab_size

    def forward(
        self,
        src: torch.Tensor,
        tgt: torch.Tensor,    
    ) -> torch.Tensor:
        
        batch_size = src.shape[0]
        target_length = tgt.shape[1]

        encoder_output, hidden, cell = self.encoder(src)

        # Start with the first target token as the first input which is <SOS>
        input_token = tgt[:, 0]

        all_logits = torch.zeros(
            batch_size,
            tgt.shape[1] - 1,
            self.target_vocab_size,
            device=src.device
        )

        # First time step t=1, input token is still at index 0 for <SOS>
        for t in range(1, target_length):
            context, _ = self.bahdanau(hidden[-1], encoder_output)
            
            logits, hidden, cell = self.decoder(
                input_token,
                context,
                hidden,
                cell
            )
            
            all_logits[:, t - 1, :] = logits
            
            input_token = tgt[:, t]

        # Return all logits for each time position, for each sequence in the batch
        return all_logits
        
    # TODO(Cedric): Implement early stopping if generate EOS token
    def generate(
        self,
        src: torch.Tensor,
        sos_idx: int,
        max_length: int,
    ) -> torch.Tensor:
        
        batch_size = src.shape[0]

        input_token = torch.full(
            (batch_size,),
            sos_idx,
            dtype=torch.long,
            device=src.device
        )

        encoder_output, hidden, cell = self.encoder(src)

        generated_steps = []

        for _ in range(max_length):
            context, _ = self.bahdanau(hidden[-1], encoder_output)
            
            logits, hidden, cell = self.decoder(
                input_token,
                context,
                hidden,
                cell,
            )

            input_token = logits.argmax(dim=-1)
            generated_steps.append(input_token)

        return torch.stack(generated_steps, dim=1)



In [115]:
model = Seq2Seq(
    encoder=encoder,
    decoder=decoder,
    bahdanau=bahdanau,
    target_vocab_size=TGT_VOCAB_SIZE,
)

out = model.generate(src, SOS_IDX, TGT_LENGTH)

assert out.shape == (BATCH_SIZE, TGT_LENGTH)

out.shape

torch.Size([5, 3])